### Ingestión del archivo "file_genre.csv"

In [0]:
%run '../includes/configuration'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

#####Paso 1 - Crear esquema y leer el archivo csv

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
genres_schema = StructType(fields=[
    StructField("genreId", IntegerType(), False),
    StructField("genreName", StringType(), True)
])

In [0]:
genres_df = spark.read \
    .option("header",True) \
    .schema(genres_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")

##### Paso 2 - Seleccionar solo las columnas requeridas

In [0]:
genres_renamed_df = genres_df \
    .withColumnRenamed("genreId", "genre_id") \
    .withColumnRenamed("genreName","genre_name")

#####Paso 4 - Agregar la columna ingestion_date y environment

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
genres_final_df = genres_renamed_df \
                    .withColumn("ingestion_date", current_timestamp()).withColumn("environment",lit(v_environment)) \
                    .withColumn("file_date",lit(v_file_date))
                

##### Paso 5 - Escribir en silver

In [0]:
genres_final_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("movie_silver.genres") 

In [0]:
%sql
SELECT * FROM movie_silver.genres

genre_id,genre_name,ingestion_date,environment,file_date
12,Adventure,2026-11-04T21:10:55.986854Z,production,2024-12-16
14,Fantasy,2026-11-04T21:10:55.986854Z,production,2024-12-16
16,Animation,2026-11-04T21:10:55.986854Z,production,2024-12-16
18,Drama,2026-11-04T21:10:55.986854Z,production,2024-12-16
27,Horror,2026-11-04T21:10:55.986854Z,production,2024-12-16
28,Action,2026-11-04T21:10:55.986854Z,production,2024-12-16
35,Comedy,2026-11-04T21:10:55.986854Z,production,2024-12-16
36,History,2026-11-04T21:10:55.986854Z,production,2024-12-16
37,Western,2026-11-04T21:10:55.986854Z,production,2024-12-16
53,Thriller,2026-11-04T21:10:55.986854Z,production,2024-12-16
